# Concurrency: Threading and Multiprocessing

**Concurrency** = doing multiple things over the same period of time.

Python has three approaches:

| Approach | Best For | Module |
|----------|----------|--------|
| **Threading** | I/O-bound tasks (waiting for network/disk) | `threading` |
| **Multiprocessing** | CPU-bound tasks (heavy computation) | `multiprocessing` |
| **Async/Await** | Many concurrent I/O operations | `asyncio` |

### The GIL (Global Interpreter Lock)
CPython has a GIL that allows only **one thread to execute Python bytecode at a time**. This means:
- Threads are great for I/O (threads release GIL while waiting)
- Threads don't speed up CPU-bound code
- Use multiprocessing for true CPU parallelism

## 1. Threading — I/O-Bound Concurrency

In [ ]:
import threading
import time

def download(url, delay):
    """Simulate downloading a URL (I/O wait)."""
    print(f'Starting download: {url}')
    time.sleep(delay)  # simulate network wait
    print(f'Finished: {url} ({delay}s)')

urls = [
    ('https://example.com/file1', 2),
    ('https://example.com/file2', 1),
    ('https://example.com/file3', 3),
]

# Sequential
start = time.time()
for url, delay in urls:
    download(url, delay)
print(f'Sequential: {time.time() - start:.2f}s\n')

# Concurrent with threads
start = time.time()
threads = []
for url, delay in urls:
    t = threading.Thread(target=download, args=(url, delay))
    threads.append(t)
    t.start()

for t in threads:
    t.join()  # wait for all threads to complete

print(f'Threaded: {time.time() - start:.2f}s')  # ~3s instead of 6s

## 2. Thread Safety — Locks

When multiple threads access shared data, you need a **Lock** to prevent race conditions.

In [ ]:
import threading

# Without lock — race condition!
counter_unsafe = 0

def increment_unsafe():
    global counter_unsafe
    for _ in range(10000):
        counter_unsafe += 1  # NOT atomic!

threads = [threading.Thread(target=increment_unsafe) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()
print(f'Unsafe counter: {counter_unsafe}  (expected 50000)')  # likely wrong!

# With lock — thread-safe
counter_safe = 0
lock = threading.Lock()

def increment_safe():
    global counter_safe
    for _ in range(10000):
        with lock:  # only one thread can run this block at a time
            counter_safe += 1

threads = [threading.Thread(target=increment_safe) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()
print(f'Safe counter: {counter_safe}  (expected 50000)')  # always correct

## 3. `concurrent.futures` — High-Level Interface

`ThreadPoolExecutor` and `ProcessPoolExecutor` are the recommended way to run concurrent tasks.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def fetch_data(source):
    """Simulate fetching data."""
    delay = {'api1': 1, 'api2': 2, 'api3': 0.5, 'api4': 1.5}.get(source, 1)
    time.sleep(delay)
    return f'Data from {source} (took {delay}s)'

sources = ['api1', 'api2', 'api3', 'api4']

start = time.time()
# max_workers=4 means 4 threads simultaneously
with ThreadPoolExecutor(max_workers=4) as executor:
    # submit() starts a task and returns a Future
    futures = {executor.submit(fetch_data, s): s for s in sources}
    
    # as_completed yields futures as they finish
    for future in as_completed(futures):
        source = futures[future]
        result = future.result()
        print(result)

print(f'Total time: {time.time() - start:.2f}s')  # ~2s instead of 5s

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# map() is simpler when you want all results in order
def square(n):
    time.sleep(0.01)  # simulate work
    return n ** 2

import time
numbers = list(range(20))

start = time.time()
with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(square, numbers))
print(f'Results: {results}')
print(f'Time: {time.time() - start:.2f}s')

## 4. Multiprocessing — CPU-Bound Parallelism

Multiprocessing spawns separate **processes**, each with its own Python interpreter and GIL. True parallel execution on multiple CPU cores.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import time

def cpu_heavy(n):
    """CPU-intensive task: find primes."""
    primes = []
    for num in range(2, n):
        if all(num % i != 0 for i in range(2, int(num**0.5) + 1)):
            primes.append(num)
    return len(primes)

tasks = [50000, 50000, 50000, 50000]

# Sequential
start = time.time()
results_seq = [cpu_heavy(n) for n in tasks]
seq_time = time.time() - start
print(f'Sequential: {seq_time:.2f}s — results: {results_seq}')

# Parallel with ProcessPoolExecutor
start = time.time()
with ProcessPoolExecutor() as executor:
    results_par = list(executor.map(cpu_heavy, tasks))
par_time = time.time() - start
print(f'Parallel:   {par_time:.2f}s — results: {results_par}')
print(f'Speedup: {seq_time/par_time:.1f}x')

## 5. Thread-Safe Data Structures

In [ ]:
import threading
import queue

# queue.Queue is thread-safe
work_queue = queue.Queue()
results = []
results_lock = threading.Lock()

def worker():
    while True:
        item = work_queue.get()
        if item is None:  # sentinel value — signal to stop
            break
        result = item ** 2  # process item
        with results_lock:
            results.append(result)
        work_queue.task_done()

# Start workers
workers = [threading.Thread(target=worker) for _ in range(3)]
for w in workers:
    w.start()

# Feed work
for i in range(10):
    work_queue.put(i)

# Send stop signals
for _ in workers:
    work_queue.put(None)

for w in workers:
    w.join()

print('Results:', sorted(results))

## Common Mistakes

### Mistake 1: Using Threading for CPU-Bound Work (GIL Blocks You)

In [ ]:
import threading, time

def cpu_bound(n):
    # Pure computation — GIL is held
    return sum(range(n))

# Threading does NOT speed up CPU-bound work due to GIL
start = time.time()
threads = [threading.Thread(target=cpu_bound, args=(5_000_000,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
thread_time = time.time() - start

start = time.time()
for _ in range(4):
    cpu_bound(5_000_000)
seq_time = time.time() - start

print(f'Sequential: {seq_time:.2f}s')
print(f'Threaded:   {thread_time:.2f}s  ← not faster due to GIL!')
print('For CPU work: use multiprocessing or ProcessPoolExecutor instead')

## Mini-Project: Parallel File Downloader Simulation

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import time
import random

# Shared progress tracker
progress_lock = threading.Lock()
downloaded = []

def simulate_download(file_info):
    """Simulate downloading a file."""
    filename, size_mb = file_info
    # Simulate speed: 5-15 MB/s
    speed = random.uniform(5, 15)
    duration = size_mb / speed
    
    time.sleep(duration)  # simulate download
    
    with progress_lock:
        downloaded.append(filename)
    
    return {'file': filename, 'size_mb': size_mb, 'duration': duration}


files = [
    ('report_q1.pdf', 5),
    ('dataset.csv', 50),
    ('presentation.pptx', 10),
    ('video_clip.mp4', 200),
    ('config.json', 0.1),
    ('archive.zip', 80),
]

print(f'Downloading {len(files)} files with 3 concurrent connections...\n')
start = time.time()

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(simulate_download, f): f for f in files}
    
    for future in as_completed(futures):
        result = future.result()
        print(f'✓ {result["file"]:25} {result["size_mb"]:6.1f} MB  in {result["duration"]:.1f}s')

total_time = time.time() - start
total_size = sum(f[1] for f in files)
print(f'\nAll done! {total_size:.1f} MB in {total_time:.1f}s')
print(f'Effective speed: {total_size/total_time:.1f} MB/s')